# SmartCrop AI - PlantDoc Experiment E (Low-Overfitting Ensemble)

**Goal:** reduce the train/val gap and push validation accuracy toward 85%.

**What changed vs Experiment D (all legitimate, no val/test leakage into training):**
- 3-model ensemble (ConvNeXt-S @384, ConvNeXt-B @384, ConvNeXt-S @320) with softmax averaging
- Lighter augmentation: MixUp only (no CutMix, it crops lesions), MIX_PROB 0.3, no RandomErasing, crop scale (0.6, 1)
- Stronger *model* regularization instead: drop_path 0.3, weight decay 0.1, label smoothing 0.1
- Layer-wise LR decay (0.8) so early layers change less -> less memorization
- EMA weights, early stopping on val accuracy
- Exact-duplicate check across train/val/test
- Test split evaluated once, at the very end

Honest note: val accuracy is used for checkpoint/TTA selection, so it is slightly optimistic. Test accuracy is the number to report.

In [1]:
# Cell 1: Setup + config
import os, json, time, copy, math, random, urllib.parse, hashlib
from pathlib import Path
from typing import Any, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support

try:
    import timm
except ImportError:
    os.system("pip install -q timm")
    import timm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

# Ensemble members. If a model runs out of memory: lower bs and raise accum (effective batch = bs*accum).
CONFIGS = [
    dict(name="convnext_small.fb_in22k_ft_in1k",     res=384, bs=16, accum=1, seed=42),
    dict(name="convnext_base.fb_in22k_ft_in1k_384",  res=384, bs=8,  accum=2, seed=7),
    dict(name="convnext_small.fb_in22k_ft_in1k",     res=320, bs=16, accum=1, seed=123),
]

EPOCHS = 30
WARMUP_EPOCHS = 2
PATIENCE = 10
LR_TOP = 1e-4          # LR of the last backbone stage
LR_HEAD = 1e-3
LLRD = 0.8             # layer-wise LR decay
WEIGHT_DECAY = 0.1
DROP_PATH = 0.3
LABEL_SMOOTHING = 0.1
MIX_PROB = 0.3         # MixUp only, gentle
MIXUP_ALPHA = 0.2
EMA_DECAY = 0.998
NUM_WORKERS = 4
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


print("Device:", DEVICE, "| Members:", [(c["name"].split(".")[0], c["res"]) for c in CONFIGS])

Device: cuda | Members: [('convnext_small', 384), ('convnext_base', 384), ('convnext_small', 320)]


In [2]:
# Cell 2: Paths + manifest + class mapping (same as Experiment D)
def locate_plantdoc_dataset_root() -> Path:
    candidates = [Path("/kaggle/input/plantdoc-dataset"), Path("/kaggle/input/plantdoc"),
                  Path("/kaggle/input/plantdoc-dataset-master"), Path("/kaggle/input/plantdoc-data"),
                  Path("datasets/plantdoc"), Path("../datasets/plantdoc")]
    for c in candidates:
        if c.exists() and (c / "train").exists():
            return c.resolve()
    ki = Path("/kaggle/input")
    if ki.exists():
        for sub in ki.rglob("*"):
            if sub.is_dir() and (sub / "train").exists() and (sub / "test").exists():
                return sub.resolve()
    for sub in Path(".").rglob("*"):
        if sub.is_dir() and sub.name == "plantdoc" and (sub / "train").exists():
            return sub.resolve()
    raise FileNotFoundError("PlantDoc dataset directory not found.")


def locate_file_path(filename, explicit):
    for c in explicit:
        if c.exists():
            return c.resolve()
    ki = Path("/kaggle/input")
    if ki.exists():
        m = list(ki.rglob(filename))
        if m:
            return m[0].resolve()
    m = list(Path(".").rglob(filename))
    if m:
        return m[0].resolve()
    raise FileNotFoundError(filename)


DATASET_ROOT = locate_plantdoc_dataset_root()
MANIFEST_PATH = locate_file_path("plantdoc_split_manifest.json", [
    Path("/kaggle/input/datasets/anyasreekadali/plantdoc-support-files/plantdoc_split_manifest.json"),
    Path("/kaggle/input/plantdoc-support-files/plantdoc_split_manifest.json"),
    Path("outputs/plantdoc_split_manifest.json")])
CLASSES_PATH = locate_file_path("disease_classes.json", [
    Path("/kaggle/input/datasets/anyasreekadali/plantdoc-support-files/disease_classes.json"),
    Path("/kaggle/input/plantdoc-support-files/disease_classes.json"),
    Path("models/disease/disease_classes.json")])

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest = json.load(f)
with open(CLASSES_PATH, "r", encoding="utf-8") as f:
    raw_classes = json.load(f)

cls_to_idx, idx_to_cls = {}, {}
if isinstance(raw_classes, dict):
    if next(iter(raw_classes.keys())).isdigit():
        for k, v in raw_classes.items():
            cls_to_idx[str(v).strip()] = int(k); idx_to_cls[int(k)] = str(v).strip()
    else:
        for k, v in raw_classes.items():
            cls_to_idx[str(k).strip()] = int(v); idx_to_cls[int(v)] = str(k).strip()
else:
    for i, n in enumerate(raw_classes):
        cls_to_idx[str(n).strip()] = i; idx_to_cls[i] = str(n).strip()

num_classes = len(idx_to_cls)
assert num_classes == 27
names = [idx_to_cls[i] for i in range(num_classes)]


def resolve_entry_metadata(entry: Any) -> Tuple[str, str]:
    if isinstance(entry, dict):
        rel = entry.get("file") or entry.get("path") or entry.get("image") or ""
        cls = entry.get("class") or entry.get("label") or entry.get("disease") or ""
        if not cls and rel:
            parts = [p for p in str(rel).replace("\\", "/").split("/") if p]
            if len(parts) >= 2:
                cls = parts[-2]
        return str(rel).strip(), str(cls).strip()
    rel = str(entry).strip()
    parts = [p for p in rel.replace("\\", "/").split("/") if p]
    return rel, (parts[-2].strip() if len(parts) >= 2 else "")


def build_records(name, entries):
    out, bad = [], 0
    for entry in entries:
        rel, cls = resolve_entry_metadata(entry)
        if not rel or cls not in cls_to_idx:
            bad += 1; continue
        norm = rel.replace("\\", "/")
        cands = [DATASET_ROOT / norm, DATASET_ROOT / rel, DATASET_ROOT / norm.replace("&", "_"),
                 DATASET_ROOT / urllib.parse.unquote(norm),
                 DATASET_ROOT / urllib.parse.unquote(norm).replace("&", "_")]
        found = next((c for c in cands if c.exists() and c.is_file()), None)
        if found is None:
            bad += 1; continue
        out.append((str(found), cls_to_idx[cls], cls))
    print(f"[{name.upper()}] mapped={len(out)} skipped={bad}")
    if bad:
        raise RuntimeError(f"{bad} unresolved entries in {name}")
    return out


train_records = build_records("train", manifest["train"])
val_records = build_records("val", manifest["val"])
test_records = build_records("test", manifest["test"])
assert (len(train_records), len(val_records), len(test_records)) == (1783, 383, 383)

train_labels = [r[1] for r in train_records]
class_counts = [train_labels.count(c) for c in range(num_classes)]
_cc = torch.tensor(class_counts, dtype=torch.float32)
CLASS_W = (1.0 / _cc.sqrt()); CLASS_W = (CLASS_W / CLASS_W.mean()).to(DEVICE)

# Exact-duplicate check across splits (byte-identical files)
def md5(p):
    return hashlib.md5(open(p, "rb").read()).hexdigest()

h = {k: {md5(r[0]) for r in v} for k, v in [("train", train_records), ("val", val_records), ("test", test_records)]}
print("Exact duplicates  train&val:", len(h["train"] & h["val"]),
      "| train&test:", len(h["train"] & h["test"]), "| val&test:", len(h["val"] & h["test"]))

[TRAIN] mapped=1783 skipped=0
[VAL] mapped=383 skipped=0
[TEST] mapped=383 skipped=0
Exact duplicates  train&val: 0 | train&test: 0 | val&test: 0


In [3]:
# Cell 3: Data, model helpers, EMA, LLRD
class PlantDocDataset(Dataset):
    def __init__(self, records, transform=None):
        self.records, self.transform = records, transform
    def __len__(self):
        return len(self.records)
    def __getitem__(self, i):
        path, label, _ = self.records[i]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label


def make_loaders(res, bs):
    pre = res + 32
    train_tf = transforms.Compose([
        transforms.Resize((pre, pre)),
        transforms.RandomResizedCrop(res, scale=(0.6, 1.0), ratio=(0.85, 1.18)),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomVerticalFlip(0.3),
        transforms.TrivialAugmentWide(),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((pre, pre)), transforms.CenterCrop(res),
        transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    pin = DEVICE.type == "cuda"
    kw = dict(num_workers=NUM_WORKERS, pin_memory=pin)
    tr = DataLoader(PlantDocDataset(train_records, train_tf), batch_size=bs, shuffle=True, drop_last=True,
                    persistent_workers=NUM_WORKERS > 0, **kw)
    tr_ev = DataLoader(PlantDocDataset(train_records, eval_tf), batch_size=bs, shuffle=False, **kw)
    va = DataLoader(PlantDocDataset(val_records, eval_tf), batch_size=bs, shuffle=False, **kw)
    te = DataLoader(PlantDocDataset(test_records, eval_tf), batch_size=bs, shuffle=False, **kw)
    return tr, tr_ev, va, te


class EMA:
    def __init__(self, model, decay):
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)
        self.decay, self.step = decay, 0

    @torch.no_grad()
    def update(self, model):
        self.step += 1
        d = min(self.decay, (1 + self.step) / (10 + self.step))
        msd = model.state_dict()
        for k, v in self.module.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(d).add_(msd[k].detach(), alpha=1 - d)
            else:
                v.copy_(msd[k])


def llrd_param_groups(model):
    """stem=id0, stages.0..3=id1..4, head=head LR. LR = LR_TOP * LLRD^(4-id)."""
    groups = {}
    for n, p in model.named_parameters():
        if n.startswith("stem"):
            lid = 0
        elif n.startswith("stages."):
            lid = int(n.split(".")[1]) + 1
        else:
            lid = 5
        lr = LR_HEAD if lid == 5 else LR_TOP * (LLRD ** (4 - lid))
        wd = 0.0 if p.ndim <= 1 else WEIGHT_DECAY
        groups.setdefault((lid, wd), {"params": [], "lr": lr, "base_lr": lr, "weight_decay": wd})["params"].append(p)
    return list(groups.values())


@torch.no_grad()
def get_probs(net, loader, tta=False):
    net.eval()
    P, Y = [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            p = net(x).float().softmax(-1)
            if tta:
                p = (p + net(torch.flip(x, dims=[3])).float().softmax(-1)
                       + net(torch.flip(x, dims=[2])).float().softmax(-1)) / 3
        P.append(p.cpu().numpy()); Y.append(y.numpy())
    return np.concatenate(P), np.concatenate(Y)


def acc(p, y):
    return float((p.argmax(1) == y).mean())


def macro_f1(p, y):
    return float(precision_recall_fscore_support(y, p.argmax(1), average="macro", zero_division=0)[2])

In [4]:
# Cell 4: Train one ensemble member (val used ONLY for checkpoint selection; test untouched)
def train_one(cfg):
    set_seed(cfg["seed"])
    res, bs, accum = cfg["res"], cfg["bs"], cfg.get("accum", 1)
    tr_l, trev_l, va_l, te_l = make_loaders(res, bs)
    key = f"{cfg['name'].split('.')[0]}@{res}_s{cfg['seed']}"

    model = timm.create_model(cfg["name"], pretrained=True, num_classes=num_classes,
                              drop_path_rate=DROP_PATH).to(DEVICE).to(memory_format=torch.channels_last)
    opt = torch.optim.AdamW(llrd_param_groups(model))
    steps_per_epoch = len(tr_l) // accum
    total_steps, warmup_steps = EPOCHS * steps_per_epoch, WARMUP_EPOCHS * steps_per_epoch

    def lr_factor(s):
        if s < warmup_steps:
            return (s + 1) / warmup_steps
        pr = (s - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * pr))

    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    ema = EMA(model, EMA_DECAY)
    best_acc, best_f1, best_ep, best_state, no_imp, step = 0.0, 0.0, -1, None, 0, 0
    print("=" * 78); print("TRAINING", key); print("=" * 78)

    for ep in range(1, EPOCHS + 1):
        t0 = time.time(); model.train(); opt.zero_grad(set_to_none=True); run, n = 0.0, 0
        for i, (x, y) in enumerate(tr_l):
            x = x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
            y = y.to(DEVICE, non_blocking=True)
            mix = np.random.rand() < MIX_PROB
            if mix:
                lam = float(np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA))
                idx = torch.randperm(x.size(0), device=x.device)
                x = lam * x + (1 - lam) * x[idx]; ya, yb = y, y[idx]
            with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                logits = model(x)
            logits = logits.float()
            ce = lambda t: F.cross_entropy(logits, t, weight=CLASS_W, label_smoothing=LABEL_SMOOTHING)
            loss = lam * ce(ya) + (1 - lam) * ce(yb) if mix else ce(y)
            scaler.scale(loss / accum).backward()
            if (i + 1) % accum == 0:
                for g in opt.param_groups:
                    g["lr"] = g["base_lr"] * lr_factor(step)
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
                ema.update(model); step += 1
            run += loss.item() * x.size(0); n += x.size(0)

        pr, yv = get_probs(model, va_l); pe, _ = get_probs(ema.module, va_l)
        use_ema = (acc(pe, yv), macro_f1(pe, yv)) >= (acc(pr, yv), macro_f1(pr, yv))
        cp, net = (pe, ema.module) if use_ema else (pr, model)
        a, f = acc(cp, yv), macro_f1(cp, yv)
        print(f"[{key} Ep {ep:02d}/{EPOCHS}] loss {run/n:.4f} | val raw {acc(pr,yv)*100:.2f}% EMA {acc(pe,yv)*100:.2f}% | {time.time()-t0:.0f}s")
        if a > best_acc + 1e-5 or (abs(a - best_acc) <= 1e-5 and f > best_f1):
            best_acc, best_f1, best_ep, no_imp = a, f, ep, 0
            best_state = copy.deepcopy(net.state_dict())
            print(f"  --> best ({'ema' if use_ema else 'raw'}) ep {ep}: val {a*100:.2f}%")
        else:
            no_imp += 1
            if no_imp >= PATIENCE:
                print("Early stopping."); break

    model.load_state_dict(best_state)
    out = {"key": key, "best_epoch": best_ep}
    for tta in (False, True):
        out[f"train_{tta}"], out["y_train"] = get_probs(model, trev_l, tta)
        out[f"val_{tta}"], out["y_val"] = get_probs(model, va_l, tta)
        out[f"test_{tta}"], out["y_test"] = get_probs(model, te_l, tta)
    torch.save(best_state, f"expE_{key}.pth")
    del model, ema, opt; torch.cuda.empty_cache()
    return out


results = []
for cfg in CONFIGS:
    results.append(train_one(cfg))
    r = results[-1]
    print(f"== {r['key']}: train {acc(r['train_False'], r['y_train'])*100:.2f}% | "
          f"val {acc(r['val_False'], r['y_val'])*100:.2f}% | test {acc(r['test_False'], r['y_test'])*100:.2f}%")

model.safetensors: reconstructing file:   0%|          |  0.00B /  201MB            

model.safetensors: downloading bytes:           |  0.00B            

TRAINING convnext_small@384_s42
[convnext_small@384_s42 Ep 01/30] loss 3.0555 | val raw 53.52% EMA 54.05% | 66s
  --> best (ema) ep 1: val 54.05%
[convnext_small@384_s42 Ep 02/30] loss 1.9676 | val raw 70.50% EMA 72.58% | 58s
  --> best (ema) ep 2: val 72.58%
[convnext_small@384_s42 Ep 03/30] loss 1.6426 | val raw 71.28% EMA 74.67% | 59s
  --> best (ema) ep 3: val 74.67%
[convnext_small@384_s42 Ep 04/30] loss 1.5370 | val raw 73.63% EMA 75.20% | 59s
  --> best (ema) ep 4: val 75.20%
[convnext_small@384_s42 Ep 05/30] loss 1.4291 | val raw 75.46% EMA 74.67% | 59s
  --> best (raw) ep 5: val 75.46%
[convnext_small@384_s42 Ep 06/30] loss 1.3310 | val raw 75.46% EMA 76.24% | 58s
  --> best (ema) ep 6: val 76.24%
[convnext_small@384_s42 Ep 07/30] loss 1.3013 | val raw 74.93% EMA 77.28% | 58s
  --> best (ema) ep 7: val 77.28%
[convnext_small@384_s42 Ep 08/30] loss 1.2152 | val raw 75.46% EMA 79.90% | 58s
  --> best (ema) ep 8: val 79.90%
[convnext_small@384_s42 Ep 09/30] loss 1.1446 | val raw 

model.safetensors: reconstructing file:   0%|          |  0.00B /  354MB            

model.safetensors: downloading bytes:           |  0.00B            

TRAINING convnext_base@384_s7
[convnext_base@384_s7 Ep 01/30] loss 2.7805 | val raw 62.92% EMA 63.97% | 103s
  --> best (ema) ep 1: val 63.97%
[convnext_base@384_s7 Ep 02/30] loss 1.6051 | val raw 71.28% EMA 75.72% | 100s
  --> best (ema) ep 2: val 75.72%
[convnext_base@384_s7 Ep 03/30] loss 1.4242 | val raw 75.46% EMA 76.24% | 100s
  --> best (ema) ep 3: val 76.24%
[convnext_base@384_s7 Ep 04/30] loss 1.3222 | val raw 76.50% EMA 75.20% | 100s
  --> best (raw) ep 4: val 76.50%
[convnext_base@384_s7 Ep 05/30] loss 1.2369 | val raw 75.46% EMA 75.98% | 100s
[convnext_base@384_s7 Ep 06/30] loss 1.1494 | val raw 77.28% EMA 79.11% | 100s
  --> best (ema) ep 6: val 79.11%
[convnext_base@384_s7 Ep 07/30] loss 1.0855 | val raw 77.02% EMA 79.37% | 100s
  --> best (ema) ep 7: val 79.37%
[convnext_base@384_s7 Ep 08/30] loss 1.0472 | val raw 76.24% EMA 78.85% | 100s
[convnext_base@384_s7 Ep 09/30] loss 1.0062 | val raw 79.37% EMA 79.90% | 100s
  --> best (ema) ep 9: val 79.90%
[convnext_base@384_s7

In [5]:
# Cell 5: Ensemble audit (test evaluated once, here)
def ens(split, tta):
    return np.mean([r[f"{split}_{tta}"] for r in results], axis=0)

yt, yv, yte = results[0]["y_train"], results[0]["y_val"], results[0]["y_test"]
val_plain, val_tta = acc(ens("val", False), yv), acc(ens("val", True), yv)
use_tta = val_tta > val_plain            # decided on VALIDATION only
P_tr, P_va, P_te = ens("train", use_tta), ens("val", use_tta), ens("test", use_tta)

tr_a, va_a, te_a = acc(P_tr, yt), acc(P_va, yv), acc(P_te, yte)
print("=" * 64)
print(f"ENSEMBLE of {len(results)} models | TTA used: {use_tta}")
print("=" * 64)
for r in results:
    print(f"  {r['key']:<34s} train {acc(r['train_False'], yt)*100:5.2f} | val {acc(r['val_False'], yv)*100:5.2f} | test {acc(r['test_False'], yte)*100:5.2f}")
print("-" * 64)
print(f"  Train acc (clean) : {tr_a*100:.2f}%")
print(f"  Val acc           : {va_a*100:.2f}%  macro-F1 {macro_f1(P_va, yv)*100:.2f}%  [{'PASS' if va_a >= 0.85 else 'below 85%'}]")
print(f"  Test acc          : {te_a*100:.2f}%  macro-F1 {macro_f1(P_te, yte)*100:.2f}%  (held-out)")
print(f"  Train-Val gap     : {(tr_a-va_a)*100:.2f} points   (Exp D was 19.86)")
print("=" * 64)

report = classification_report(yv, P_va.argmax(1), target_names=names, output_dict=True, zero_division=0)
print(classification_report(yv, P_va.argmax(1), target_names=names, zero_division=0))
cm = confusion_matrix(yv, P_va.argmax(1))
pairs = sorted([(cm[i, j], names[i], names[j]) for i in range(num_classes) for j in range(num_classes) if i != j and cm[i, j] > 0], reverse=True)[:10]
print("Top confusions (true -> predicted):")
for n, t, p in pairs:
    print(f"  {n:2d}x  {t} -> {p}")

json.dump({"experiment": "Experiment_E_Ensemble", "members": [r["key"] for r in results], "tta_used": bool(use_tta),
           "train_acc": tr_a, "val_acc": va_a, "val_macro_f1": macro_f1(P_va, yv),
           "test_acc": te_a, "test_macro_f1": macro_f1(P_te, yte), "gap_train_val": tr_a - va_a,
           "per_class_val_report": report}, open("plantdoc_experiment_E_metrics.json", "w"), indent=2)
np.savez("plantdoc_experiment_E_probs.npz", train=P_tr, val=P_va, test=P_te, y_train=yt, y_val=yv, y_test=yte)
print("Saved: plantdoc_experiment_E_metrics.json, plantdoc_experiment_E_probs.npz, expE_*.pth")

ENSEMBLE of 3 models | TTA used: False
  convnext_small@384_s42             train 94.67 | val 79.90 | test 81.20
  convnext_base@384_s7               train 98.99 | val 81.46 | test 81.72
  convnext_small@320_s123            train 98.71 | val 80.16 | test 81.72
----------------------------------------------------------------
  Train acc (clean) : 98.60%
  Val acc           : 80.42%  macro-F1 79.02%  [below 85%]
  Test acc          : 83.55%  macro-F1 82.43%  (held-out)
  Train-Val gap     : 18.18 points   (Exp D was 19.86)
                            precision    recall  f1-score   support

           Apple Scab Leaf       0.83      0.71      0.77        14
                Apple leaf       0.88      1.00      0.93        14
           Apple rust leaf       0.92      0.92      0.92        13
          Bell_pepper leaf       0.86      0.67      0.75         9
     Bell_pepper leaf spot       0.79      1.00      0.88        11
            Blueberry leaf       0.94      1.00      0.97       